# NeurIPS Revision Session 1 - Fresh Kaggle Run

Zero-dollar GPU notebook. Start from a fresh Kaggle notebook with **Internet ON** and **GPU ON**.

This session runs:

- Fix 1 generation + analysis
- Fix 5 coherence-preserving noise slope
- Fix 11 RAPTOR full table

Expected runtime on a free Kaggle GPU: roughly 4-9 hours.

## 1. Clone Repo And Check GPU

Run this first. If this fails, check Kaggle Internet and GPU settings.

In [ ]:
%%bash
set -euo pipefail

REPO_URL="https://github.com/Saket-Maganti/rag-hallucination-detection.git"
REPO_DIR="/kaggle/working/rag-hallucination-detection"

cd /kaggle/working
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone --branch main "$REPO_URL" "$REPO_DIR"
else
  git -C "$REPO_DIR" fetch origin main
  git -C "$REPO_DIR" checkout main
  git -C "$REPO_DIR" pull --ff-only origin main
fi

cd "$REPO_DIR"
git log -1 --oneline
echo "repo path: $PWD"
nvidia-smi || true
python --version
df -h /kaggle/working

## 2. Install Ollama And Pull Mistral

This installs `zstd` first because Kaggle images may not include it and Ollama needs it for extraction.

In [ ]:
%%bash
set -euo pipefail

REPO_DIR="/kaggle/working/rag-hallucination-detection"
if [ ! -d "$REPO_DIR/.git" ]; then
  cd /kaggle/working
  git clone --branch main https://github.com/Saket-Maganti/rag-hallucination-detection.git "$REPO_DIR"
fi
cd "$REPO_DIR"

apt-get update -y
apt-get install -y zstd curl git zip

if ! command -v ollama >/dev/null 2>&1; then
  curl -fsSL https://ollama.com/install.sh | sh
fi

if ! pgrep -x ollama >/dev/null 2>&1; then
  nohup ollama serve > /kaggle/working/ollama.log 2>&1 &
fi

for i in $(seq 1 30); do
  if ollama list >/tmp/ollama_list.txt 2>&1; then
    break
  fi
  sleep 2
done

ollama pull mistral
ollama list

## 3. Install Python Dependencies

This intentionally avoids paid/API-specific packages and does not depend on notebook cwd.

In [ ]:
%%bash
set -euo pipefail

REPO_DIR="/kaggle/working/rag-hallucination-detection"
if [ ! -d "$REPO_DIR/.git" ]; then
  cd /kaggle/working
  git clone --branch main https://github.com/Saket-Maganti/rag-hallucination-detection.git "$REPO_DIR"
fi
cd "$REPO_DIR"

python -m pip install -U pip setuptools wheel
python -m pip install -e pip-package
python -m pip install \
  pandas numpy scipy scikit-learn matplotlib seaborn tabulate tqdm \
  datasets sentence-transformers transformers torch accelerate \
  langchain langchain-community langchain-core langchain-text-splitters langchain-ollama \
  chromadb

python - <<'PY'
import pandas, numpy, scipy, sklearn, torch, transformers, datasets
print('python deps OK')
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
PY

## 4. Preflight

This should print `preflight OK` before you start long runs.

In [ ]:
%%bash
set -euo pipefail

REPO_DIR="/kaggle/working/rag-hallucination-detection"
if [ ! -d "$REPO_DIR/.git" ]; then
  cd /kaggle/working
  git clone --branch main https://github.com/Saket-Maganti/rag-hallucination-detection.git "$REPO_DIR"
fi
cd "$REPO_DIR"
mkdir -p logs/revision

python -m py_compile \
  experiments/fix_01_causal_matched_pairs.py \
  experiments/fix_05_coherence_preserving_noise.py \
  experiments/fix_11_raptor_full_table.py \
  experiments/revision_utils.py \
  src/ragas_scorer.py src/vectara_hem_scorer.py

python experiments/fix_01_causal_matched_pairs.py --help >/tmp/fix01_help.txt
python experiments/fix_05_coherence_preserving_noise.py --help >/tmp/fix05_help.txt
python experiments/fix_11_raptor_full_table.py --help >/tmp/fix11_help.txt
ollama list
echo "preflight OK"

## 5. Fix 1 - Generation And Analysis

Expected runtime: 45-120 minutes on a free GPU. If `matched_pairs.csv` is missing, this cell reconstructs it first.

In [ ]:
%%bash
set -euo pipefail

REPO_DIR="/kaggle/working/rag-hallucination-detection"
if [ ! -d "$REPO_DIR/.git" ]; then
  cd /kaggle/working
  git clone --branch main https://github.com/Saket-Maganti/rag-hallucination-detection.git "$REPO_DIR"
fi
cd "$REPO_DIR"
mkdir -p logs/revision

if [ ! -s data/revision/fix_01/matched_pairs.csv ]; then
  PYTHONUNBUFFERED=1 python -u experiments/fix_01_causal_matched_pairs.py \
    --stage construct \
    --dataset squad \
    --n_target 200 \
    --seed 42 \
    --max_contexts 400 \
    --candidate_limit 400 \
    --run_tag primary_n200 \
    2>&1 | tee logs/revision/fix_01_construct_session1.log
fi

PYTHONUNBUFFERED=1 python -u experiments/fix_01_causal_matched_pairs.py \
  --stage generate \
  --backend ollama \
  --model mistral \
  --resume \
  --save_every 2 \
  --progress_every 10 \
  2>&1 | tee logs/revision/fix_01_generate_session1.log

PYTHONUNBUFFERED=1 python -u experiments/fix_01_causal_matched_pairs.py \
  --stage analyze \
  2>&1 | tee logs/revision/fix_01_analyze_session1.log

## 6. Fix 5 - Coherence-Preserving Noise Slope

Expected runtime: 1.5-4 hours on a free GPU.

In [ ]:
%%bash
set -euo pipefail

REPO_DIR="/kaggle/working/rag-hallucination-detection"
if [ ! -d "$REPO_DIR/.git" ]; then
  cd /kaggle/working
  git clone --branch main https://github.com/Saket-Maganti/rag-hallucination-detection.git "$REPO_DIR"
fi
cd "$REPO_DIR"
mkdir -p logs/revision

PYTHONUNBUFFERED=1 python -u experiments/fix_05_coherence_preserving_noise.py \
  --n 200 \
  --seed 42 \
  --backend ollama \
  --model mistral \
  --max_contexts 300 \
  --n_noise 1 2 3 \
  --save_every 25 \
  2>&1 | tee logs/revision/fix_05_noise_slope_session1.log

## 7. Fix 11 - RAPTOR Full Table

Expected runtime: 1-2.5 hours on a free GPU.

In [ ]:
%%bash
set -euo pipefail

REPO_DIR="/kaggle/working/rag-hallucination-detection"
if [ ! -d "$REPO_DIR/.git" ]; then
  cd /kaggle/working
  git clone --branch main https://github.com/Saket-Maganti/rag-hallucination-detection.git "$REPO_DIR"
fi
cd "$REPO_DIR"
mkdir -p logs/revision

PYTHONUNBUFFERED=1 python -u experiments/fix_11_raptor_full_table.py \
  --datasets squad pubmedqa hotpotqa \
  --n 100 \
  --backend ollama \
  --model mistral \
  --max_contexts 150 \
  --raptor_clusters 6 \
  2>&1 | tee logs/revision/fix_11_raptor_full_table_session1.log

## 8. Package Outputs

Download `/kaggle/working/revision_session1_outputs.zip` after this cell finishes.

In [ ]:
%%bash
set -euo pipefail

REPO_DIR="/kaggle/working/rag-hallucination-detection"
cd "$REPO_DIR"
rm -f /kaggle/working/revision_session1_outputs.zip

zip -r /kaggle/working/revision_session1_outputs.zip \
  data/revision/fix_01 results/revision/fix_01 \
  data/revision/fix_05 results/revision/fix_05 \
  data/revision/fix_11 results/revision/fix_11 \
  logs/revision \
  REVISION_SUMMARY.md REVISION_RUNBOOK.md

ls -lh /kaggle/working/revision_session1_outputs.zip